## Modelo de predicción — v20


In [ ]:
!pip install uv
!uv pip install -r requirements.txt

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn provee las herramientas de machine learning
from sklearn.preprocessing import StandardScaler           # normalización de variables
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor         # el modelo principal
from sklearn.metrics import r2_score, mean_squared_error  # métricas de evaluación

import os

## Carga de datos

In [ ]:
# Cargar cada fuente de datos por separado — se combinarán en la siguiente celda
water_quality_df       = pd.read_csv('water_quality_training_dataset.csv')
landsat_train_features = pd.read_csv('landsat_features_training_v4.csv')
Terraclimate_df        = pd.read_csv('terraclimate_features_training_v3.csv')
q_train_df             = pd.read_csv('tc_q_train.csv')

# GLHYMPS contiene permeabilidad del suelo (logK) y porosidad — se carga si existe
glhymps_df = pd.read_csv('glhymps_features_train.csv') if os.path.exists('glhymps_features_train.csv') else None
print('GLHYMPS:', glhymps_df.shape if glhymps_df is not None else 'no disponible')

print(f'WQ:      {water_quality_df.shape}')
print(f'Landsat: {landsat_train_features.shape}')
print(f'TC v3:   {Terraclimate_df.shape}')
print(f'Q:       {q_train_df.shape}')

# Verificar que los datasets están alineados fila a fila por coordenadas
# Un porcentaje bajo indicaría que los archivos están en diferente orden
lat_ls = (water_quality_df['Latitude'].values == landsat_train_features['Latitude'].values).mean()
lat_tc = (water_quality_df['Latitude'].values == Terraclimate_df['Latitude'].values).mean()
print(f'Alineacion Landsat: {lat_ls:.1%}  |  TC v3: {lat_tc:.1%}')
assert lat_ls > 0.95, f'ERROR: Landsat desalineado ({lat_ls:.1%})'
assert lat_tc > 0.95, f'ERROR: TC v3 desalineado ({lat_tc:.1%})'
print('OK: datasets alineados')

## Combinación y feature engineering

In [ ]:
def combine_datasets(d1, d2, d3):
    # pd.concat con axis=1 une columnas horizontalmente (no apila filas)
    data = pd.concat([d1, d2, d3], axis=1)
    # Eliminar columnas duplicadas que pueden aparecer al unir (ej. Latitude repetida)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_datasets(water_quality_df, landsat_train_features, Terraclimate_df)
# Q se añade por separado porque se extrajo en un notebook independiente
wq_data['q'] = q_train_df['q'].values

# Incorporar GLHYMPS si está disponible — geología del substrato
if glhymps_df is not None:
    assert len(glhymps_df) == len(wq_data), f'GLHYMPS desfase: {len(glhymps_df)} vs {len(wq_data)}'
    for col in ['logK', 'porosity']:
        if col in glhymps_df.columns:
            wq_data[col] = glhymps_df[col].values
    print(f'GLHYMPS añadido: logK corr_TA={wq_data["logK"].corr(wq_data["Total Alkalinity"]):.3f}')

print(f'Shape combinado: {wq_data.shape}')

In [ ]:
# Clipping: limitar valores a rangos físicamente posibles para eliminar outliers extremos
# Los valores fuera de rango suelen ser errores de medición o artefactos del sensor

# Bandas Landsat: valores de reflectancia digital (0-60000 es el rango válido en C2-L2)
for col in ['nir', 'green', 'red', 'blue', 'swir16', 'swir22']:
    if col in wq_data.columns:
        wq_data[col] = wq_data[col].clip(upper=60000)

# Índices normalizados de agua: por definición están entre -1 y 1
# Se restringe a un rango más conservador ya que valores extremos son ruido
for col in ['NDMI', 'MNDWI', 'NDTI']:
    if col in wq_data.columns:
        wq_data[col] = wq_data[col].clip(-0.5, 0.5)

# Índices de tierra: también normalizados entre -1 y 1
for col in ['ndvi_land', 'ndbi', 'bsi']:
    if col in wq_data.columns:
        wq_data[col] = wq_data[col].clip(-1.0, 1.0)

# GLHYMPS: rangos geológicos válidos (logK en escala logarítmica, porosidad en fracción)
if 'logK'     in wq_data.columns: wq_data['logK']     = wq_data['logK'].clip(-16, -3)
if 'porosity' in wq_data.columns: wq_data['porosity'] = wq_data['porosity'].clip(0, 0.5)

print('Clipping OK')

In [ ]:
eps = 1e-9  # valor mínimo para evitar divisiones por cero

# Diferencia entre dos bandas infrarrojas — sensible a turbidez y contenido mineral
wq_data['swir_diff'] = wq_data['swir16'] - wq_data['swir22']

# Extraer el mes de la fecha de muestra
wq_data['date']  = pd.to_datetime(wq_data['Sample Date'], dayfirst=True, errors='coerce')
wq_data['month'] = wq_data['date'].dt.month

# Codificación cíclica del mes usando seno y coseno:
# Esto evita que el modelo trate enero (1) y diciembre (12) como distantes,
# cuando en realidad están a un mes de diferencia en el ciclo anual
wq_data['month_sin'] = np.sin(2 * np.pi * wq_data['month'] / 12)
wq_data['month_cos'] = np.cos(2 * np.pi * wq_data['month'] / 12)

print('Features derivadas OK')

In [ ]:
# Calcular medianas del set de entrenamiento — se usarán también para imputar la validación
# Se usa mediana (no media) porque es más robusta ante valores extremos
train_medians = wq_data.median(numeric_only=True)
wq_data = wq_data.fillna(train_medians)
print(f'NaNs restantes: {wq_data.isna().sum().sum()}')

In [ ]:
# Umbral mínimo de correlación absoluta con al menos uno de los tres targets
# Una variable con |r| < 0.08 con todos los targets aporta más ruido que señal al modelo
CORR_THRESHOLD = 0.08

# Lista completa de variables candidatas (incluyendo GLHYMPS si está disponible)
FEATURE_COLS_ALL = [
    'nir', 'green', 'red', 'blue', 'swir22', 'swir_diff',
    'NDMI', 'MNDWI', 'NDTI',
    'pet', 'ppt', 'soil', 'aet', 'water_deficit', 'q',
    'month_sin', 'month_cos',
    'ndvi_land', 'ndbi', 'bsi',
    'logK', 'porosity',
]

TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Filtrar las que realmente existen en el dataset cargado
available = [c for c in FEATURE_COLS_ALL if c in wq_data.columns]

# Calcular correlación de cada variable con cada target y quedarse con el máximo
corr_abs = wq_data[available + TARGET_COLS].corr()[TARGET_COLS].loc[available].abs()
max_corr  = corr_abs.max(axis=1)
FEATURE_COLS = [f for f in available if max_corr[f] >= CORR_THRESHOLD]
dropped      = [f for f in available if max_corr[f] < CORR_THRESHOLD]

print(f'Features activas ({len(FEATURE_COLS)}): {FEATURE_COLS}')
if dropped:
    print(f'Eliminadas por corr < {CORR_THRESHOLD} ({len(dropped)}): {dropped}')

wq_data_model = wq_data[FEATURE_COLS + TARGET_COLS]
print(f'Shape: {wq_data_model.shape}')

## Diagnóstico: correlación de features con targets

In [ ]:
# Tabla de correlaciones: muestra cuánto se relaciona cada variable con cada target
# Rojo = correlación positiva fuerte, Azul = negativa fuerte, blanco = sin relación
corr = wq_data_model.corr()[TARGET_COLS].loc[FEATURE_COLS]
print('Correlación features vs targets:')
display(corr.round(3))

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación features vs targets')
plt.tight_layout()
plt.show()

## Pipeline de modelado

In [ ]:
from sklearn.model_selection import GridSearchCV

def split_data(X, y, test_size=0.3, random_state=42):
    """Divide el dataset en 70% entrenamiento y 30% prueba de forma aleatoria y reproducible."""
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    """
    Normaliza las variables: resta la media y divide por la desviación estándar.
    El scaler se ajusta SOLO con datos de entrenamiento — luego se aplica igual a test y validación.
    Esto evita que información de test influya en el entrenamiento.
    """
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test), scaler

def evaluate_model(model, X_scaled, y_true, dataset_name='Test', log_target=False):
    """Predice y calcula R² y RMSE. Si el target fue transformado con log1p, deshace la transformación."""
    y_pred = model.predict(X_scaled)
    if log_target:
        y_pred = np.expm1(y_pred)  # inversa de log1p: e^x - 1
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'  {dataset_name}: R²={r2:.4f}  RMSE={rmse:.4f}')
    return y_pred, r2, rmse

def select_features(X_train_scaled, y_train, feature_names, importance_threshold=0.5):
    """
    Entrena un Random Forest rápido (100 árboles) y elimina las variables con importancia
    menor al threshold * media de importancias. Garantiza un mínimo de 8 variables.
    """
    selector = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    selector.fit(X_train_scaled, y_train)
    importances = selector.feature_importances_
    threshold   = importance_threshold * importances.mean()
    if sum(imp >= threshold for imp in importances) < 8:
        # Si el umbral elimina demasiadas variables, forzar al menos 8
        threshold = sorted(importances, reverse=True)[8]
    selected  = [f for f, imp in zip(feature_names, importances) if imp >= threshold]
    eliminated = [f for f, imp in zip(feature_names, importances) if imp < threshold]
    print(f'  Features seleccionadas ({len(selected)}): {selected}')
    if eliminated:
        print(f'  Eliminadas ({len(eliminated)}): {eliminated}')
    return selected

def run_pipeline(X_df, y, feature_cols, param_name='Parameter', log_transform=False,
                 n_estimators=500, min_samples_leaf=5, max_features=0.6,
                 run_gridsearch=False):
    """
    Pipeline completo para un target: split → normalización → selección de features
    → (opcional) GridSearch → entrenamiento → validación cruzada → evaluación.

    log_transform=True aplica log1p al target antes de entrenar — útil cuando la distribución
    tiene cola larga (como DRP), ya que estabiliza la varianza y mejora el ajuste.
    """
    print(f"\n{'='*65}")
    print(f'  {param_name}')
    print(f"{'='*65}")

    y_fit = np.log1p(y) if log_transform else y.copy()
    X = X_df[feature_cols]
    X_train, X_test, y_train, y_test = split_data(X, y_fit)
    y_test_orig = y.iloc[y_test.index] if log_transform else y_test
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

    print('\n[Feature Selection]')
    selected_features = select_features(X_train_scaled, y_train, feature_cols, importance_threshold=0.25)
    feat_idx    = [feature_cols.index(f) for f in selected_features]
    X_train_sel = X_train_scaled[:, feat_idx]
    X_test_sel  = X_test_scaled[:,  feat_idx]

    if run_gridsearch:
        # GridSearch evalúa todas las combinaciones de hiperparámetros con validación cruzada
        # n_estimators: número de árboles | min_samples_leaf: mínimo de muestras por hoja
        # max_features: fracción de variables consideradas en cada split
        print('\n[GridSearch]')
        param_grid = {
            'n_estimators':      [500, 700, 1000],
            'min_samples_leaf':  [3, 5, 8, 12],
            'max_features':      [0.4, 0.5, 0.6, 0.7],
        }
        gs = GridSearchCV(
            RandomForestRegressor(random_state=42, n_jobs=-1),
            param_grid,
            cv=KFold(n_splits=5, shuffle=True, random_state=42),
            scoring='r2',
            n_jobs=-1,
            verbose=0,
        )
        gs.fit(X_train_sel, y_train)
        best = gs.best_params_
        print(f'  Mejores params: {best}')
        n_estimators     = best['n_estimators']
        min_samples_leaf = best['min_samples_leaf']
        max_features     = best['max_features']

    print(f'\n  RF: n_est={n_estimators}  min_leaf={min_samples_leaf}  max_feat={max_features}')

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1  # usar todos los núcleos disponibles
    )
    model.fit(X_train_sel, y_train)

    # Validación cruzada: divide el entrenamiento en 5 partes y evalúa 5 veces
    # Esto da una estimación más fiable del rendimiento que una sola división
    print('\n[Cross-Validation 5-Fold]')
    cv_scores = cross_val_score(
        RandomForestRegressor(
            n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
            max_features=max_features, random_state=42, n_jobs=-1
        ),
        X_train_sel, y_train,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring='r2'
    )
    print(f'  CV R² por fold: {cv_scores.round(4)}')
    print(f'  CV R² mean±std: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

    print('\n[Evaluación]')
    if log_transform:
        y_train_pred_orig = np.expm1(model.predict(X_train_sel))
        y_train_orig      = np.expm1(y_train.values)
        r2_train   = r2_score(y_train_orig, y_train_pred_orig)
        rmse_train = np.sqrt(mean_squared_error(y_train_orig, y_train_pred_orig))
        print(f'  Train: R²={r2_train:.4f}  RMSE={rmse_train:.4f}')
        _, r2_test, rmse_test = evaluate_model(model, X_test_sel, y_test_orig, 'Test', log_target=True)
    else:
        _, r2_train, rmse_train = evaluate_model(model, X_train_sel, y_train, 'Train')
        _, r2_test,  rmse_test  = evaluate_model(model, X_test_sel,  y_test,  'Test')

    results = {
        'Parameter':      param_name,
        'CV_R2_mean':     cv_scores.mean(),
        'CV_R2_std':      cv_scores.std(),
        'R2_Train':       r2_train,
        'RMSE_Train':     rmse_train,
        'R2_Test':        r2_test,
        'RMSE_Test':      rmse_test,
        'n_features_sel': len(selected_features),
        'n_estimators':   n_estimators,
        'min_leaf':       min_samples_leaf,
        'max_features':   max_features,
    }
    return model, scaler, selected_features, pd.DataFrame([results]), log_transform

In [ ]:
X_df = wq_data_model[FEATURE_COLS]

y_TA  = wq_data_model['Total Alkalinity']
y_EC  = wq_data_model['Electrical Conductance']
y_DRP = wq_data_model['Dissolved Reactive Phosphorus']

# Se entrena un modelo independiente por cada target
# DRP usa log_transform=True porque su distribución tiene cola muy pronunciada
# run_gridsearch=True hace la búsqueda automática de hiperparámetros (~5-10 min total)
# Si ya se conocen los mejores params, se puede poner False y ajustarlos manualmente
model_TA, scaler_TA, feat_TA, results_TA, log_TA = run_pipeline(
    X_df, y_TA, FEATURE_COLS, 'Total Alkalinity',
    log_transform=False, run_gridsearch=True,
)

model_EC, scaler_EC, feat_EC, results_EC, log_EC = run_pipeline(
    X_df, y_EC, FEATURE_COLS, 'Electrical Conductance',
    log_transform=False, run_gridsearch=True,
)

model_DRP, scaler_DRP, feat_DRP, results_DRP, log_DRP = run_pipeline(
    X_df, y_DRP, FEATURE_COLS, 'Dissolved Reactive Phosphorus',
    log_transform=True, run_gridsearch=True,
)

In [ ]:
# Resumen comparativo de los tres modelos
# CV_R2_mean es el indicador más fiable — es la métrica de validación cruzada
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
display(results_summary[['Parameter','CV_R2_mean','CV_R2_std','R2_Train','R2_Test','RMSE_Test','n_features_sel']])

print('\nFeatures seleccionadas por modelo:')
print(f'  TA:  {feat_TA}')
print(f'  EC:  {feat_EC}')
print(f'  DRP: {feat_DRP}')

In [ ]:
# Visualización de la importancia de cada variable en los tres modelos
# La importancia indica cuánto contribuye cada variable a las decisiones del bosque
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, model, feat_list, name in zip(
    axes,
    [model_TA, model_EC, model_DRP],
    [feat_TA, feat_EC, feat_DRP],
    ['Total Alkalinity', 'Electrical Conductance', 'DRP']
):
    imps   = model.feature_importances_
    idx    = np.argsort(imps)[::-1]  # ordenar de mayor a menor importancia
    colors = ['steelblue' if imps[i] > imps.mean() else 'lightgray' for i in idx]
    ax.bar(range(len(feat_list)), imps[idx], color=colors)
    ax.set_xticks(range(len(feat_list)))
    ax.set_xticklabels([feat_list[i] for i in idx], rotation=45, ha='right', fontsize=7)
    ax.axhline(imps.mean(), color='red', linestyle='--', alpha=0.4)  # línea de importancia media
    ax.set_title(name)
    ax.set_ylabel('Importancia')

plt.suptitle('Feature Importance por modelo', fontsize=11)
plt.tight_layout()
plt.show()

## Predicción sobre dataset de validación

In [ ]:
# Cargar los datos de validación — los mismos archivos pero con las filas a predecir
test_file            = pd.read_csv('submission_template.csv')
landsat_val_features = pd.read_csv('landsat_features_validation_v4.csv')
Terraclimate_val_df  = pd.read_csv('terraclimate_features_validation_v3.csv')
q_val_df             = pd.read_csv('tc_q_val.csv')

glhymps_val_df = pd.read_csv('glhymps_features_val.csv') if os.path.exists('glhymps_features_val.csv') else None
print('GLHYMPS val:', glhymps_val_df.shape if glhymps_val_df is not None else 'no disponible')

print(f'Template:    {test_file.shape}')
print(f'Landsat val: {landsat_val_features.shape}')
print(f'TC v3 val:   {Terraclimate_val_df.shape}')

# Verificar alineación — crítico para que las predicciones correspondan a los puntos correctos
lat_ls = (test_file['Latitude'].values == landsat_val_features['Latitude'].values).mean()
lat_tc = (test_file['Latitude'].values == Terraclimate_val_df['Latitude'].values).mean()
print(f'Alineacion Landsat val: {lat_ls:.1%}  |  TC v3 val: {lat_tc:.1%}')
assert lat_ls > 0.95, f'ERROR Landsat val desalineado ({lat_ls:.1%})'
assert lat_tc > 0.95, f'ERROR TC v3 val desalineado ({lat_tc:.1%})'
print('OK: validacion alineada')

In [ ]:
eps = 1e-9

# Construir tabla de validación con exactamente las mismas transformaciones que el entrenamiento
# Es crucial que cada variable se procese de forma idéntica para que el modelo funcione bien
val_data = pd.DataFrame({
    'Latitude':    landsat_val_features['Latitude'].values,
    'Longitude':   landsat_val_features['Longitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
})

# Bandas espectrales de Landsat
for col in ['blue', 'green', 'red', 'nir', 'swir16', 'swir22']:
    if col in landsat_val_features.columns:
        val_data[col] = landsat_val_features[col].values

# Índices espectrales calculados durante la extracción
for col in ['NDMI', 'MNDWI', 'NDTI', 'ndvi_land', 'ndbi', 'bsi']:
    if col in landsat_val_features.columns:
        val_data[col] = landsat_val_features[col].values

# Variables climáticas de TerraClimate
val_data['pet']           = Terraclimate_val_df['pet'].values
val_data['ppt']           = Terraclimate_val_df['ppt'].values
val_data['soil']          = Terraclimate_val_df['soil'].values
val_data['aet']           = Terraclimate_val_df['aet'].values
val_data['water_deficit'] = Terraclimate_val_df['water_deficit'].values
val_data['q']             = q_val_df['q'].values

if glhymps_val_df is not None:
    for col in ['logK', 'porosity']:
        if col in glhymps_val_df.columns:
            val_data[col] = glhymps_val_df[col].values

# Aplicar exactamente el mismo clipping que en entrenamiento
for col in ['nir','green','red','blue','swir16','swir22']:
    if col in val_data.columns: val_data[col] = val_data[col].clip(upper=60000)
for col in ['NDMI','MNDWI','NDTI']:
    if col in val_data.columns: val_data[col] = val_data[col].clip(-0.5, 0.5)
for col in ['ndvi_land','ndbi','bsi']:
    if col in val_data.columns: val_data[col] = val_data[col].clip(-1.0, 1.0)
if 'logK'     in val_data.columns: val_data['logK']     = val_data['logK'].clip(-16, -3)
if 'porosity' in val_data.columns: val_data['porosity'] = val_data['porosity'].clip(0, 0.5)

# Calcular features derivadas con la misma lógica que en entrenamiento
val_data['swir_diff'] = val_data['swir16'] - val_data['swir22']
val_data['date']      = pd.to_datetime(val_data['Sample Date'], dayfirst=True, errors='coerce')
val_data['month']     = val_data['date'].dt.month
val_data['month_sin'] = np.sin(2 * np.pi * val_data['month'] / 12)
val_data['month_cos'] = np.cos(2 * np.pi * val_data['month'] / 12)

# Imputar NaN con las medianas calculadas en el set de entrenamiento (no de validación)
val_data = val_data.fillna(train_medians)

print(f'Val shape: {val_data.shape}')
print(f'NaNs: {val_data.isna().sum().sum()}')

# Comparar distribuciones para detectar posible distribución-shift entre train y val
print(f'\nDistribucion train vs val:')
for col in ['soil', 'pet', 'q']:
    print(f'  {col:12s} train: mean={wq_data[col].mean():.2f}  std={wq_data[col].std():.2f}  '
          f'| val: mean={val_data[col].mean():.2f}  std={val_data[col].std():.2f}')

In [ ]:
def predict_submission(model, scaler, feat_list, all_feature_cols, val_df, log_target=False):
    """
    Aplica el mismo scaler del entrenamiento a los datos de validación,
    selecciona las features usadas por este modelo y genera predicciones.
    Si el target fue transformado con log1p, aplica la inversa (expm1).
    """
    X_all    = scaler.transform(val_df[all_feature_cols])
    feat_idx = [all_feature_cols.index(f) for f in feat_list]
    X_sel    = X_all[:, feat_idx]
    preds    = model.predict(X_sel)
    if log_target:
        preds = np.expm1(preds)         # deshacer log1p: obtener valores originales
        preds = np.clip(preds, 0, None) # asegurar que no haya predicciones negativas
    return preds

pred_TA  = predict_submission(model_TA,  scaler_TA,  feat_TA,  FEATURE_COLS, val_data, log_target=log_TA)
pred_EC  = predict_submission(model_EC,  scaler_EC,  feat_EC,  FEATURE_COLS, val_data, log_target=log_EC)
pred_DRP = predict_submission(model_DRP, scaler_DRP, feat_DRP, FEATURE_COLS, val_data, log_target=log_DRP)

print(f'TA  — min:{pred_TA.min():.2f}   max:{pred_TA.max():.2f}   mean:{pred_TA.mean():.2f}')
print(f'EC  — min:{pred_EC.min():.2f}   max:{pred_EC.max():.2f}   mean:{pred_EC.mean():.2f}')
print(f'DRP — min:{pred_DRP.min():.5f}  max:{pred_DRP.max():.5f}  mean:{pred_DRP.mean():.5f}')

In [ ]:
# Construir el DataFrame de submission con las predicciones de los tres targets
submission_df = pd.DataFrame({
    'Latitude':                      test_file['Latitude'].values,
    'Longitude':                     test_file['Longitude'].values,
    'Sample Date':                   test_file['Sample Date'].values,
    'Total Alkalinity':              pred_TA,
    'Electrical Conductance':        pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP,
})

# Reordenar columnas exactamente como el template — el sistema evaluador lee por posición
submission_df = submission_df[test_file.columns.tolist()]

print('Columnas:', submission_df.columns.tolist())
print(f'Orden OK: {submission_df.columns.tolist() == test_file.columns.tolist()}')
print(f'\nPredicciones:')
print(f'  TA  — min:{pred_TA.min():.2f}  max:{pred_TA.max():.2f}  mean:{pred_TA.mean():.2f}  std:{pred_TA.std():.2f}')
print(f'  EC  — min:{pred_EC.min():.2f}  max:{pred_EC.max():.2f}  mean:{pred_EC.mean():.2f}  std:{pred_EC.std():.2f}')
print(f'  DRP — min:{pred_DRP.min():.4f}  max:{pred_DRP.max():.4f}  mean:{pred_DRP.mean():.4f}  std:{pred_DRP.std():.4f}')
display(submission_df.head(5))

In [ ]:
SUBMISSION_FILE = '/tmp/submission_v20.csv'

submission_df.to_csv(SUBMISSION_FILE, index=False)

# Verificar que el header del CSV guardado coincide exactamente con el template
# Una diferencia de orden causaría un score incorrecto en la plataforma
with open(SUBMISSION_FILE) as f:
    header = f.readline().strip()
    row1   = f.readline().strip()
print(f'Header: {header}')
print(f'Row 1:  {row1[:100]}')

expected = ','.join(test_file.columns.tolist())
assert header == expected, f'ERROR header CSV:\n  got:      {header}\n  expected: {expected}'
print('OK: header correcto')

# PUT sube el archivo local al almacenamiento interno de Snowflake
result = session.sql(f"""
    PUT file://{SUBMISSION_FILE}
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print(f'PUT result: {result}')

# Listar el workspace para confirmar que el archivo quedó disponible
files = session.sql("""
    LIST 'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
""").collect()
for row in files:
    if 'submission' in str(row).lower():
        print(f'En Snowflake: {row}')